# Open in Colab
<a target="_blank" href="https://colab.research.google.com/github/Nicolepcx/ai-agents-the-definitive-guide/blob/main/CH01/ch01_code_examples.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# About this notebook

This notebook is a compact, end to end primer on turning a plain LLM call into a small but real agent that can use tools, keep state across turns, and trace its own reasoning flow.

## What it shows

* **Stateless LLM call** with `langchain-openai` to warm up.
* **Tool use in a loop** with two simple tools

  1. `internet_search` via SerpAPI for fresh information
  2. `calculator` powered by `numexpr` for fast math.
* **A minimal LangGraph agent** with

  * a typed state that accumulates `messages` using `add_messages`
  * an `llm` node bound to tools and a `ToolNode` for execution
  * a router that decides when to call tools or stop
  * in memory checkpoints via `MemorySaver`
  * separate threads to branch conversations and compare memory.

## Why these examples

1. You see the **bare minimum loop** that handles tool calls without any framework state.
2. You then see the **same idea done right** with LangGraph, so you get clean routing, checkpoints, and thread scoped memory.
3. You get **trace utilities** to print node updates and a short state snapshot, which makes debugging and teaching much easier.

## How to run it

* Load keys from `.env` or set them with `%env`. If missing, the notebook will ask interactively.
* Call `gpt-5-mini` or any other LLM you like compatible with OAI API, once for a quick check, then switch to `gpt-4o` bound to tools.
* Run a **two step task**:
  * get the **current air temperature in New York City** using `internet_search`
  * **square the temperature** with `calculator`.
* Repeat the task inside a **LangGraph** app, then branch a **second thread** that converts the temperature to Fahrenheit and reports both.
  You will see per node updates and a final memory snapshot for each thread.

## Key ideas to notice

* **Binding tools to the model** and letting the model choose when to call them.
* A **minimal router** that checks for `tool_calls` and either transitions to the `ToolNode` or ends.
* **Thread ids** for independent memories and reproducible debugging.
* **Deterministic setups** where helpful
  `temperature=0`, explicit `recursion_limit`, and short format prompts to produce consistent outputs.

## Swap and extend

* You can swap `SerpAPIWrapper` with any other search tool that returns text.
* Add your own tools and drop them into `tools` and `tool_map`.
* Replace `MemorySaver` with a persistent checkpointer if you want long lived sessions.
* Add nodes for planning, validation, or guardrails if you want a larger graph.

## Requirements and notes

* You need a working [`OPENAI_API_KEY`](https://platform.openai.com/api-keys) and a [`SerpAPIWrapper`](https://serpapi.com/).
* Internet search results change. The reported temperature and snippets will vary by time and source.
* Tool calls count toward tokens and external API usage. Keep an eye on cost.


# Dependencies

In [1]:
!pip install -q langgraph==0.6.7 langchain-openai==0.3.33 python-dotenv==1.1.1 langchain_community google-search-results

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.3 MB/s eta 0:00:00
ERROR: pip's depe

# Imports for API

In [2]:

from dotenv import load_dotenv
import os

# Imports

In [3]:
# Standard library
import os
import math
import json
import numexpr
from typing import List, Dict, Any, TypedDict, Annotated

# LangChain core
from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool
from langchain_community.utilities import SerpAPIWrapper

# LangGraph
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver


/usr/local/lib/python3.12/dist-packages/langgraph/checkpoint/base/__init__.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [4]:
from google.colab import userdata
import os
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["SERPAPI_API_KEY"] = userdata.get("SERPAPI_API_KEY")

In [5]:
# --- API Key Setup ---
# Option 1 (preferred): create a `.env` file in your project folder with:
# OPENAI_API_KEY=your_openai_key_here
# SERPAPI_API_KEY=your_serpapi_key_here
#
# Option 2: set it directly in the notebook with magic:
# %env OPENAI_API_KEY=your_openai_key_here
# %env SERPAPI_API_KEY=your_serpapi_key_here

from dotenv import load_dotenv
import os

# Load from .env if available
load_dotenv()

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
serp_api_key = os.getenv("SERPAPI_API_KEY")

# Fallback: ask if still missing
if not OPENAI_API_KEY:
    print("⚠️ OPENAI_API_KEY not found. You can set it with `%env` in the notebook or enter it below.")
    OPENAI_API_KEY = input("Enter your OPENAI_API_KEY: ").strip()

if not serp_api_key:
    print("⚠️ SERPAPI_API_KEY not found. You can set it with `%env` in the notebook or enter it below.")
    serp_api_key = input("Enter your SERPAPI_API_KEY: ").strip()

print("✅ API keys loaded successfully!")



✅ API keys loaded successfully!


# First Example: Stateless LLM call with LangChain
____

## Setup LLM

In [6]:
llm = ChatOpenAI(model="gpt-5-mini")
response = llm.invoke("What are AI agents?")
print(response.content)


An AI agent is a software or embodied system that perceives its environment, makes decisions, and takes actions to achieve goals. It’s the practical “actor” in AI systems: not just a model that answers questions, but something that senses, reasons, and acts (often repeatedly) to accomplish tasks.

Key ideas
- Perceive → decide → act: an agent receives input (sensors, data), uses some internal process (rules, planning, learning, optimization) to choose an action, and executes it in the environment.
- Goal-directedness: agents generally try to achieve objectives (e.g., maximize safety, profit, user satisfaction).
- Autonomy and rationality: agents operate with some independence and aim to act rationally (best action given their knowledge and objectives), though not necessarily optimally.
- Learning vs fixed behavior: some agents follow fixed rules; others learn and adapt from experience (e.g., via reinforcement learning).

Types and architectures
- Reactive agents: simple stimulus → resp

## Defining tools for a stateless run


## Tools

In [6]:
@tool("internet_search")
def internet_search(query: str) -> str:
    """Search Google via SerpAPI for up to date information."""
    serp_api_key = os.environ["SERPAPI_API_KEY"]
    params = {"engine": "google", "gl": "us", "hl": "en"}
    search = SerpAPIWrapper(params=params, serpapi_api_key=serp_api_key)
    return search.run(query)

@tool("calculator")
def calculator(expression: str) -> str:
    """Evaluate a single line mathematical expression with numexpr."""
    local_dict = {"pi": math.pi, "e": math.e}
    out = numexpr.evaluate(
        expression.strip(),
        global_dict={},
        local_dict=local_dict,
    )
    return str(out)

tools = [internet_search, calculator]

tool_map: Dict[str, Any] = {t.name: t for t in tools}

## Model bound to tool

In [7]:
llm = ChatOpenAI(model="gpt-4o").bind_tools(tools, tool_choice="any")

## Stateless single run with tool loop

In [8]:
def run_once(prompt: str, max_steps: int = 4) -> str:
    messages = [HumanMessage(content=prompt)]
    for _ in range(max_steps):
        ai: AIMessage = llm.invoke(messages)
        messages.append(ai)

        calls = getattr(ai, "tool_calls", None) or []
        if not calls:
            break

        for call in calls:
            name = call["name"]
            args = call.get("args", {})
            result = tool_map[name].invoke(args)
            messages.append(ToolMessage(
                content=str(result),
                name=name,
                tool_call_id=call["id"]
            ))
    return messages[-1].content

print(run_once("""Two step task.

Step 1: Use internet_search to get the current air temperature in New York City today. Show the exact query you used, the top source title and snippet, and extract a numeric temperature in Celsius. Return this temperature as feedback for Step 2.

Step 2: Using the Celsius value from Step 1, compute its square with calculator. Show the exact expression you used and the numeric result.

Important: Give a short final answer in this format:
Current temperature:
Square of current temperature:"""))



576


# Tools

In [10]:
@tool("internet_search")
def internet_search(query: str) -> str:
    """Search Google via SerpAPI for up to date information."""
    serp_api_key = os.environ["SERPAPI_API_KEY"]
    params = {"engine": "google", "gl": "us", "hl": "en"}
    search = SerpAPIWrapper(params=params, serpapi_api_key=serp_api_key)
    return search.run(query)

@tool("calculator")
def calculator(expression: str) -> str:
    """Evaluate a single line mathematical expression with numexpr."""
    local_dict = {"pi": math.pi, "e": math.e}
    out = numexpr.evaluate(
        expression.strip(),
        global_dict={},
        local_dict=local_dict,
    )
    return str(out)

tools = [internet_search, calculator]
tool_map: Dict[str, Any] = {t.name: t for t in tools}


# Model bound to tools

In [11]:
llm = ChatOpenAI(model="gpt-4o", temperature=0, max_tokens=800).bind_tools(tools, tool_choice="auto")

# Minimal tool loop (stateless)

In [12]:

def run_once(prompt: str, max_steps: int = 8) -> str:
    messages: List[HumanMessage | AIMessage | ToolMessage] = [HumanMessage(content=prompt)]
    last_ai: AIMessage | None = None

    for _ in range(max_steps):
        ai: AIMessage = llm.invoke(messages)
        messages.append(ai)
        last_ai = ai

        calls = getattr(ai, "tool_calls", None) or []
        if not calls:
            # Model produced a final answer
            return messages[-1].content

        # Execute tool calls and feed observations back
        for call in calls:
            name = call["name"]
            args = call.get("args", {}) or {}
            result = tool_map[name].invoke(args)
            messages.append(ToolMessage(
                content=str(result),
                name=name,
                tool_call_id=call.get("id")
            ))

    # If we exit the loop without a clean final AI message, force a wrap up
    messages.append(HumanMessage(content="""
Finish now. Give a short final answer in this exact format:

Current temperature:
Square of current temperature:
""".strip()))
    final_ai: AIMessage = llm.invoke(messages)
    return final_ai.content

print(run_once("""Two step task.

Step 1: Use internet_search to get the current air temperature in New York City today. Show the exact query you used, the top source title and snippet, and extract a numeric temperature in Celsius. Return this temperature as feedback for Step 2.

Step 2: Using the Celsius value from Step 1, compute its square with calculator. Show the exact expression you used and the numeric result.

Important: Give a short final answer in this format:
Current temperature:
Square of current temperature:"""))


Current temperature: 24°C  
Square of current temperature: 576


## Tools

In [13]:
@tool("internet_search")
def internet_search(query: str) -> str:
    """Search Google via SerpAPI for up to date information."""
    serp_api_key = os.environ["SERPAPI_API_KEY"]
    params = {"engine": "google", "gl": "us", "hl": "en"}
    search = SerpAPIWrapper(params=params, serpapi_api_key=serp_api_key)
    return search.run(query)

@tool("calculator")
def calculator(expression: str) -> str:
    """Evaluate a single line mathematical expression with numexpr."""
    print(f"tool_call: calculator(expresion={expression})")
    local_dict = {"pi": math.pi, "e": math.e}
    out = numexpr.evaluate(
        expression.strip(),
        global_dict={},
        local_dict=local_dict,
    )
    return str(out)

tools = [internet_search, calculator]


## Build a minimal LangGraph with state, nodes, and routing

In [14]:
class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]

# LLM
llm = ChatOpenAI(model="gpt-4o", temperature=0, max_tokens=800).bind_tools(tools)

def llm_node(state: AgentState) -> AgentState:
    ai = llm.invoke(state["messages"])
    return {"messages": [ai]}

tool_node = ToolNode(tools=tools)

graph = StateGraph(AgentState)
graph.add_node("llm", llm_node)
graph.add_node("tools", tool_node)
graph.add_edge(START, "llm")

def route(state: AgentState):
    last = state["messages"][-1]
    calls = getattr(last, "tool_calls", None) or []
    return "tools" if calls else END

graph.add_conditional_edges("llm", route, {"tools": "tools", END: END})
graph.add_edge("tools", "llm")

## Compile with in-memory checkpoints and configure a thread

## Trace execution and inspect state and memory

In [21]:
def _short(msg: BaseMessage, max_len: int = 140) -> str:
    """Compact one-line view of a message."""
    role = type(msg).__name__.replace("Message", "").lower()
    content = getattr(msg, "content", "")
    if isinstance(content, list):
        # some tool outputs can be list payloads
        try:
            content = json.dumps(content)
        except Exception:
            content = str(content)
    text = str(content).replace("\n", " ").strip()
    if len(text) > max_len:
        text = text[: max_len - 3] + "..."
    # include tool name or function call info when available
    if hasattr(msg, "tool_calls") and getattr(msg, "tool_calls"):
        print(f"{[tc.get("args", "no args") for tc in msg.tool_calls]}")
        tnames = [tc.get("name", "tool") for tc in msg.tool_calls]
        return f"{role}: tool_calls -> {tnames}"
    if isinstance(msg, ToolMessage):
        return f"{role}({msg.name}): {text}"
    return f"{role}: {text}"

## Trace execution and inspect state and memory

In [37]:
def print_state_snapshot(app, config, title: str):
    """Print current graph state and memory for a given thread."""
    snap = app.get_state(config)
    values = snap.values or {}
    msgs: List[BaseMessage] = values.get("messages", [])
    print(f"\n=== {title} | state snapshot ===")
    print(f"messages: {len(msgs)} total")
    for i, m in enumerate(msgs[-10:], start=max(0, len(msgs)-10) + 1):
        print(f"  {i:>3}: {_short(m)}")
    # show routing info and queued tasks if present
    nxt = getattr(snap, "next", None)
    tasks = getattr(snap, "tasks", None)
    if nxt:
        print(f"next nodes: {list(nxt)}")
    if tasks:
        print(f"queued tasks: {tasks}")
    # minimal memory view via checkpointer for this thread
    # MemorySaver keeps one latest checkpoint per thread by default, so show existence
    print("memory: in-memory checkpoint present for this thread")

## Trace execution and inspect state and memory

In [18]:
def run_with_tracing(app, input_state: AgentState, config, title: str):
    """Run the graph while printing per-node updates and final memory."""
    print(f"\n=== {title} | execution trace ===")
    final = None
    # stream_mode="updates" surfaces node-level updates
    for event in app.stream(input_state, config=config, stream_mode="updates"):
        for node, upd in event.items():
            # upd is a dict like {"messages": [<new msg>]} or tool results
            keys = list(upd.keys())
            print(f"[enter {node}] updated: {keys}")
            # if messages updated, print the last one briefly
            msgs = upd.get("messages") or []
            if msgs:
                print(f"  {_short(msgs[-1])}")
            print(f"[leave {node}]")
            final = upd
    # show final assistant message from app.get_state
    print_state_snapshot(app, config, title=f"{title} | after run")
    snap = app.get_state(config)
    msgs = snap.values.get("messages", [])
    return msgs[-1].content if msgs else ""

In [28]:
checkpointer = MemorySaver()
app = graph.compile(checkpointer=checkpointer)


In [29]:
# configs
cfg = {"configurable": {"thread_id": "nyc-weather-session"}}

## Turn 1: get the current NYC air temperature in Celsius

In [30]:
turn1_answer = run_with_tracing(
    app,
    {"messages": [HumanMessage(content="Get the current air temperature in New York City in Celsius.")]},
    config={**cfg, "recursion_limit": 20},
    title="TURN 1",
)
print("\nTURN 1 (final assistant):\n", turn1_answer)


=== TURN 1 | execution trace ===
[enter llm] updated: ['messages']
[{'query': 'current air temperature in New York City in Celsius'}]
  ai: tool_calls -> ['internet_search']
[leave llm]
[enter tools] updated: ['messages']
  tool(internet_search): {'type': 'weather_result', 'temperature': '24', 'unit': 'Celsius', 'precipitation': '53%', 'humidity': '87%', 'wind': '14 km/h', 'locatio...
[leave tools]
[enter llm] updated: ['messages']
  ai: The current air temperature in New York City is 24°C.
[leave llm]

=== TURN 1 | after run | state snapshot ===
messages: 4 total
    1: human: Get the current air temperature in New York City in Celsius.
[{'query': 'current air temperature in New York City in Celsius'}]
    2: ai: tool_calls -> ['internet_search']
    3: tool(internet_search): {'type': 'weather_result', 'temperature': '24', 'unit': 'Celsius', 'precipitation': '53%', 'humidity': '87%', 'wind': '14 km/h', 'locatio...
    4: ai: The current air temperature in New York City is 24°C.
memor

## Turn 2: square that temperature in the same thread

In [31]:
turn2_answer = run_with_tracing(
    app,
    {"messages": [HumanMessage(content="Now compute the square of that temperature.")]},
    config={**cfg, "recursion_limit": 20},
    title="TURN 2",
)
print("\nTURN 2 (final assistant):\n", turn2_answer)



=== TURN 2 | execution trace ===
[enter llm] updated: ['messages']
[{'expression': '24^2'}]
  ai: tool_calls -> ['calculator']
[leave llm]
tool_call: calculator(expresion=24^2)
[enter tools] updated: ['messages']
  tool(calculator): 26
[leave tools]
[enter llm] updated: ['messages']
  ai: The square of the temperature, 24°C, is 576.
[leave llm]

=== TURN 2 | after run | state snapshot ===
messages: 8 total
    4: ai: The current air temperature in New York City is 24°C.
    5: human: Now compute the square of that temperature.
[{'expression': '24^2'}]
    6: ai: tool_calls -> ['calculator']
    7: tool(calculator): 26
    8: ai: The square of the temperature, 24°C, is 576.
memory: in-memory checkpoint present for this thread

TURN 2 (final assistant):
 The square of the temperature, 24°C, is 576.


## Branch a parallel thread for a different follow-up

In [32]:
cfg_branch = {"configurable": {"thread_id": "nyc-weather-session-branch"}}
branch_answer = run_with_tracing(
    app,
    {"messages": [HumanMessage(content="Instead of squaring, convert it to Fahrenheit and report both.")]},
    config=cfg_branch,
    title="BRANCH",
)
print("\nBRANCH (final assistant):\n", branch_answer)



=== BRANCH | execution trace ===
[enter llm] updated: ['messages']
  ai: Sure, I can help with that. Please provide the temperature in Celsius that you would like to convert to Fahrenheit.
[leave llm]

=== BRANCH | after run | state snapshot ===
messages: 2 total
    1: human: Instead of squaring, convert it to Fahrenheit and report both.
    2: ai: Sure, I can help with that. Please provide the temperature in Celsius that you would like to convert to Fahrenheit.
memory: in-memory checkpoint present for this thread

BRANCH (final assistant):
 Sure, I can help with that. Please provide the temperature in Celsius that you would like to convert to Fahrenheit.


In [34]:
# 1. Lista os checkpoints da thread original pra achar o ponto de fork
original_config = {"configurable": {"thread_id": "nyc-weather-session"}}
history = list(app.get_state_history(original_config))
for h in history:
    print(h.config["configurable"]["checkpoint_id"], h.values.get("messages", [])[-1] if h.values.get("messages") else None)

1f185566-9449-67f1-8008-c1af9f0a74ae content='The square of the temperature, 24°C, is 576.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 238, 'total_tokens': 253, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_dce478cc2a', 'id': 'chatcmpl-E4DZ2lUpRZwjIzakDQSTYEHOaKcuF', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='run--019f86de-8d4e-7383-9818-fdcab787f13d-0' usage_metadata={'input_tokens': 238, 'output_tokens': 15, 'total_tokens': 253, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
1f185566-8f0d-636c-8007-413057774fbb content='26' name='calculator' id='175e056f-1041-46a7-92a5-c0f738f1f534' tool_call_id='call_cx4YZN

In [35]:
# 2. Escolhe o checkpoint_id do ponto onde quer divergir (ex.: logo após ter a temperatura)
fork_checkpoint_id = "1f185566-8918-6994-8004-6bbc6ee721a4"  # copiado da lista acima

cfg_branch = {
    "configurable": {
        "thread_id": "nyc-weather-session",   # MESMA thread, não uma nova
        "checkpoint_id": fork_checkpoint_id
    }
}
branch_answer = run_with_tracing(app, {"messages": [HumanMessage(content="Instead of squaring, convert it to Fahrenheit and report both.")]}, config=cfg_branch, title="BRANCH")


=== BRANCH | execution trace ===
[enter llm] updated: ['messages']
[{'expression': '(24 * 9/5) + 32'}]
  ai: tool_calls -> ['calculator']
[leave llm]
tool_call: calculator(expresion=(24 * 9/5) + 32)
[enter tools] updated: ['messages']
  tool(calculator): 75.2
[leave tools]
[enter llm] updated: ['messages']
  ai: The current air temperature in New York City is 24°C, which is equivalent to 75.2°F.
[leave llm]

=== BRANCH | after run | state snapshot ===
messages: 4 total
    1: human: Get the current air temperature in New York City in Celsius.
[{'query': 'current air temperature in New York City in Celsius'}]
    2: ai: tool_calls -> ['internet_search']
    3: tool(internet_search): {'type': 'weather_result', 'temperature': '24', 'unit': 'Celsius', 'precipitation': '53%', 'humidity': '87%', 'wind': '14 km/h', 'locatio...
    4: ai: The current air temperature in New York City is 24°C.
next nodes: ['__start__']
queued tasks: (PregelTask(id='8af9e5a7-0c57-c672-9391-6127de13ac1a', name='_

In [38]:



# show consolidated memory views for both threads
print_state_snapshot(app, cfg, title="MAIN THREAD memory view")
print_state_snapshot(app, cfg_branch, title="BRANCH THREAD memory view")



=== MAIN THREAD memory view | state snapshot ===
messages: 8 total
    1: human: Get the current air temperature in New York City in Celsius.
[{'query': 'current air temperature in New York City in Celsius'}]
    2: ai: tool_calls -> ['internet_search']
    3: tool(internet_search): {'type': 'weather_result', 'temperature': '24', 'unit': 'Celsius', 'precipitation': '53%', 'humidity': '87%', 'wind': '14 km/h', 'locatio...
    4: ai: The current air temperature in New York City is 24°C.
    5: human: Instead of squaring, convert it to Fahrenheit and report both.
[{'expression': '(24 * 9/5) + 32'}]
    6: ai: tool_calls -> ['calculator']
    7: tool(calculator): 75.2
    8: ai: The current air temperature in New York City is 24°C, which is equivalent to 75.2°F.
memory: in-memory checkpoint present for this thread

=== BRANCH THREAD memory view | state snapshot ===
messages: 4 total
    1: human: Get the current air temperature in New York City in Celsius.
[{'query': 'current air temperat